# CDDFuse Variant Training Template

Template notebook để train 1 variant trên Kaggle GPU. Khi skill `fusion-train-kaggle` push notebook, các placeholder `<<...>>` sẽ được thay bằng giá trị cụ thể.

**Inputs (Kaggle Datasets, attach trước khi run):**
- `kienvbhp1234/harvard-medical-train` — 738 train pairs (CT 160 + PET 245 + SPECT 333)
- `kienvbhp1234/harvard-medical-fusion` — 72 test pairs (24×3, disjoint với train)
- `kienvbhp1234/cddfuse-pretrained` — `CDDFuse_MIF.pth`

Train/test split: Harvard 810 → 72 reference test → 738 train. Disjoint by stem.

**Outputs (`/kaggle/working/`):**
- `CDDFuse-<Variant>.pth` — variant ckpt
- `CDDFuse-<Variant>_train_history.json` — loss curves
- `CDDFuse-<Variant>/` — Fusion images + per-image metrics + summary


## Cell 1 — Config (skill auto-fills these)

In [ ]:
VARIANT     = 'FuseRule-Gated'         # <<VARIANT>>
REPO_URL    = 'https://github.com/kienvbhp872004/Image-Fusion.git'
REPO_BRANCH = 'main'                    # <<BRANCH>>
REPO_COMMIT = ''                        # <<COMMIT_SHA>>  (pin để reproducible; '' = HEAD)
EPOCHS      = 25                        # <<EPOCHS>>
BATCH       = 4                         # T4 16GB an toàn với 4; tăng 8 nếu OK
SEED        = 42

import os
os.environ['VARIANT'] = VARIANT

## Cell 2 — Clone repo, install deps

In [ ]:
!git clone --branch $REPO_BRANCH $REPO_URL /kaggle/working/Image-Fusion
%cd /kaggle/working/Image-Fusion
if REPO_COMMIT:
    !git checkout $REPO_COMMIT
!pip install -q -r requirements.txt

## Cell 3 — Unzip datasets vào layout repo

In [ ]:
import shutil, zipfile, pathlib

# Train data (738 pairs, disjoint từ test)
TRAIN_OUT = pathlib.Path('/kaggle/working/Image-Fusion/data/train')
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
for modal in ['CT-MRI', 'PET-MRI', 'SPECT-MRI']:
    src_zip = f'/kaggle/input/harvard-medical-train/{modal}.zip'
    dst_dir = TRAIN_OUT / modal
    dst_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(src_zip) as zf:
        zf.extractall(dst_dir)
    print(f'[train] {modal}: {len(list(dst_dir.rglob("*.png")))} files')

# Test data (72 pairs reference)
TEST_OUT = pathlib.Path('/kaggle/working/Image-Fusion/data/reference')
TEST_OUT.mkdir(parents=True, exist_ok=True)
for modal in ['CT-MRI', 'PET-MRI', 'SPECT-MRI']:
    src_zip = f'/kaggle/input/harvard-medical-fusion/{modal}.zip'
    dst_dir = TEST_OUT / modal
    dst_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(src_zip) as zf:
        zf.extractall(dst_dir)
    print(f'[test ] {modal}: {len(list(dst_dir.rglob("*.png")))} files')

# Pretrained ckpt
CKPT_DIR = pathlib.Path('/kaggle/working/Image-Fusion/models/MMIF-CDDFuse/models')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy('/kaggle/input/cddfuse-pretrained/CDDFuse_MIF.pth', CKPT_DIR)
print('[ckpt ] staged →', CKPT_DIR / 'CDDFuse_MIF.pth')

## Cell 4 — Train variant (light retrain, freeze encoder/decoder)

In [ ]:
%cd /kaggle/working/Image-Fusion/models/MMIF-CDDFuse
!python -m variants.train \
    --variant $VARIANT \
    --pretrained models/CDDFuse_MIF.pth \
    --train_data /kaggle/working/Image-Fusion/data/train \
    --output    /kaggle/working/ \
    --epochs    $EPOCHS \
    --batch     $BATCH \
    --seed      $SEED

## Cell 5 — Inference 3 modal trên TEST set + per-image metrics

In [ ]:
OUT_DIR = f'/kaggle/working/CDDFuse-{VARIANT}'
CKPT    = f'/kaggle/working/CDDFuse-{VARIANT}.pth'
TEST    = '/kaggle/working/Image-Fusion/data/reference'

for modal in ['CT', 'PET', 'SPECT']:
    !python evaluate_cddfuse.py \
        --variant       $VARIANT \
        --modal         $modal \
        --ckpt          $CKPT \
        --harvard_root  $TEST \
        --out_dir       $OUT_DIR \
        --save_perimage

## Cell 6 — Package output cho download

In [ ]:
import tarfile, json, hashlib, datetime

h = hashlib.sha256()
with open(CKPT, 'rb') as f:
    for chunk in iter(lambda: f.read(8192), b''):
        h.update(chunk)
ckpt_sha = h.hexdigest()

stamp = {
    'variant_name':   f'CDDFuse-{VARIANT}',
    'based_on':       'CDDFuse',
    'datetime':       datetime.datetime.utcnow().isoformat() + 'Z',
    'train_mode':     'light_retrain',
    'frozen_modules': ['Restormer_Encoder', 'Restormer_Decoder'],
    'trained_modules':['gated_b', 'gated_d', 'BaseFuseLayer', 'DetailFuseLayer'],
    'epochs_phase1':  0,
    'epochs_phase2':  EPOCHS,
    'batch_size':     BATCH,
    'seed':           SEED,
    'checkpoint':     {'path': CKPT, 'sha256': ckpt_sha},
    'data': {
        'train': 'kienvbhp1234/harvard-medical-train (738 pairs)',
        'test':  'kienvbhp1234/harvard-medical-fusion (72 pairs)',
        'split_protocol': 'Harvard 810 - 72 reference = 738 train. Disjoint by stem.'
    },
    'kaggle': {'kernel_runtime_env': 'kaggle'}
}
with open(f'{OUT_DIR}/_ablation_stamp.json', 'w') as f:
    json.dump(stamp, f, indent=2)

tar_path = f'/kaggle/working/CDDFuse-{VARIANT}_results.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    tar.add(OUT_DIR, arcname=f'CDDFuse-{VARIANT}')
    tar.add(CKPT,    arcname=f'CDDFuse-{VARIANT}.pth')
    tar.add(f'/kaggle/working/CDDFuse-{VARIANT}_train_history.json',
            arcname=f'CDDFuse-{VARIANT}_train_history.json')
print('[done]', tar_path)